# 01. Training the baseline (cross-entropy)

Frozen CLIP ViT-B/16 + Transformer decoder, trained with cross-entropy only (label smoothing 0.1) on Karpathy train + restval (113,287 images). Output: `restval_baseline_best_model.pth`.

The constants `BATCH_SIZE = 64`, `EPOCHS = 40` and `NUM_LAYERS = 3` follow the hyperparameter table of the paper. Preprocessing, augmentation, label smoothing (0.1), gradient clipping (1.0), AdamW (weight decay 0.01), the 5% linear warm-up and early stopping on validation loss (patience 5) are the same as in notebook 02.

In [ ]:
import os
import math
import re
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from collections import Counter
from transformers import CLIPVisionModel, get_linear_schedule_with_warmup
import json # json for the COCO annotations and saving the vocab

# ==========================================
# CONFIGURATION AND HYPERPARAMETERS
# ==========================================
SEED = 42

def seed_everything(seed=SEED):
    """Fix the RNG so training runs are reproducible."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True  # fixed IMAGE_SIZE/BATCH_SIZE -> safe to enable the cuDNN autotuner, faster

def seed_worker(worker_id):
    """Seed Python/NumPy in each DataLoader worker."""
    worker_seed = torch.initial_seed() % (2 ** 32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# COCO 2014 paths for the CLIP-ViT + Transformer baseline.
TRAIN_IMG_DIR = "/kaggle/input/datasets/nadaibrahim/coco2014/train2014/train2014"
VAL_IMG_DIR = "/kaggle/input/datasets/nadaibrahim/coco2014/val2014/val2014"
KARPATHY_JSON_FILE = "/kaggle/input/datasets/nganhthnh/karpathy-split/dataset_coco.json"
# Training set: Karpathy "train" (82,783 train2014 images) + "restval" (the remaining 30,504 val2014 images) = 113,287 images.
# restval does not overlap the 5,000 val / 5,000 test images. Set it back to ("train",) for the old configuration.
TRAIN_SPLITS = ("train", "restval")

# Output paths for the configuration files produced during training
OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

VOCAB_FILE = os.path.join(OUTPUT_DIR, "coco2014_karpathy_restval_baseline_vocab.json")
CHECKPOINT_FILE = os.path.join(OUTPUT_DIR, "coco2014_karpathy_restval_baseline_last_checkpoint.pth")
BEST_MODEL_FILE = os.path.join(OUTPUT_DIR, "coco2014_karpathy_restval_baseline_best_model.pth")
# Training from scratch on train+restval: old checkpoints (train only, old vocabulary) cannot be reused.
# If the kernel is interrupted, set RESUME_TRAINING=True; RESUME_CHECKPOINT_FILE points to the last_checkpoint of this restval run.
RESUME_TRAINING = False
# Set to True to resume the baseline from an old checkpoint.
# Old checkpoint (training only): "/kaggle/input/<model>/coco2014_karpathy_baseline_best_model.pth"
RESUME_CHECKPOINT_FILE = CHECKPOINT_FILE

IMAGE_SIZE = (224, 224) 
SEQ_LENGTH = 25
MIN_WORD_FREQ = 5    
EMBED_DIM = 512
FF_DIM = 512
BATCH_SIZE = 64  # paper (hyperparameter table)
EPOCHS = 40  # paper (hyperparameter table)
NUM_HEADS = 8
NUM_LAYERS = 3  # paper (hyperparameter table)
LABEL_SMOOTHING = 0.1  # same as the proposed model (notebook 02)
GRAD_CLIP_NORM = 1.0   # same as the proposed model (notebook 02)
USE_AMP = True  # mixed precision (fp16 autocast + GradScaler) for the training loop - speed only, NOT used in the val loop so val metrics stay in full fp32

PAD_IDX = 0
START_IDX = 1
END_IDX = 2
UNK_IDX = 3

# ==========================================
# DATA PREPROCESSING
# ==========================================
def custom_standardization(text):
    text = text.lower()
    strip_chars = r"""!"#$%&'()*+,-./:;=?@[\]^_`{|}~1234567890"""
    text = re.sub(f"[{re.escape(strip_chars)}]", "", text)
    return text.strip()

# Train: with augmentation (same as notebook 02)
train_transforms = T.Compose([
    T.Resize(IMAGE_SIZE),
    T.CenterCrop(IMAGE_SIZE),
    T.RandomHorizontalFlip(),
    T.RandomRotation(degrees=15),
    T.ColorJitter(contrast=0.3),
    T.ToTensor(),
    T.Normalize(mean=[0.48145466, 0.4578275, 0.40821073], 
                std=[0.26862954, 0.26130258, 0.27577711])
])

# Validation: NO augmentation
val_transforms = T.Compose([
    T.Resize(IMAGE_SIZE),
    T.CenterCrop(IMAGE_SIZE),
    T.ToTensor(),
    T.Normalize(mean=[0.48145466, 0.4578275, 0.40821073], 
                std=[0.26862954, 0.26130258, 0.27577711])
])

class COCODataset(Dataset):
    def __init__(self, image_paths, captions, vocab, transforms=None):
        self.image_paths = image_paths
        self.captions = captions
        self.vocab = vocab
        self.transforms = transforms

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        caption = self.captions[idx]
        
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception as e:
            # Fallback if an image is corrupted (very rare, but possible in a large dataset)
            image = Image.new("RGB", IMAGE_SIZE, (0, 0, 0))
            
        if self.transforms:
            image = self.transforms(image)
            
        tokens = [self.vocab.get(word, UNK_IDX) for word in caption.split()]
        tokens = [START_IDX] + tokens[:SEQ_LENGTH-2] + [END_IDX]
        
        padded_tokens = tokens + [PAD_IDX] * (SEQ_LENGTH - len(tokens))
        target_tensor = torch.tensor(padded_tokens, dtype=torch.long)
        
        return image, target_tensor

def load_karpathy_data(json_file, split="train"):
    """Read the COCO Karpathy split from dataset_coco.json. split is one name or a tuple of names, e.g. ("train", "restval")."""
    splits = {split} if isinstance(split, str) else set(split)
    print(f"Reading Karpathy split {sorted(splits)} from: {json_file}...")
    with open(json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    image_paths = []
    captions = []
    missing_images = 0

    for img in data.get('images', []):
        if img.get('split') not in splits:
            continue

        filepath = img.get('filepath', '')
        filename = img.get('filename')
        if not filename:
            continue

        image_dir = TRAIN_IMG_DIR if filepath == 'train2014' else VAL_IMG_DIR
        candidates = [
            os.path.join(image_dir, filename),
            os.path.join(os.path.dirname(image_dir), filename),
            os.path.join(os.path.dirname(image_dir), filepath, filename),
        ]
        img_path = next((path for path in candidates if os.path.exists(path)), None)
        if img_path is None:
            missing_images += 1
            continue

        for sent in img.get('sentences', []):
            if 'raw' in sent:
                caption = custom_standardization(sent['raw'])
            else:
                caption = custom_standardization(' '.join(sent.get('tokens', [])))

            if caption:
                image_paths.append(img_path)
                captions.append(caption)

    print(f"-> Karpathy {sorted(splits)}: {len(set(image_paths))} images, {len(captions)} image-caption pairs.")
    if missing_images:
        print(f"-> Skipped {missing_images} images whose files were not found.")
    return image_paths, captions

def load_coco_data(annotation_file, image_dir):
    """Read the COCO dataset JSON"""
    print(f"Reading data from: {annotation_file}...")
    with open(annotation_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    # Map image IDs to file names
    img_id_to_filename = {img['id']: img['file_name'] for img in data['images']}
    
    image_paths = []
    captions = []
    
    for ann in data['annotations']:
        img_id = ann['image_id']
        caption = custom_standardization(ann['caption'])
        
        if img_id in img_id_to_filename:
            filename = img_id_to_filename[img_id]
            img_path = os.path.join(image_dir, filename)
            
            # Only add the file if it really exists
            if os.path.exists(img_path):
                image_paths.append(img_path)
                captions.append(caption)
                
    print(f"-> Loaded {len(captions)} image-caption pairs.")
    return image_paths, captions

def build_or_load_vocab(captions, min_freq=5, vocab_file=VOCAB_FILE):
    """Build a new vocabulary, or load it from file if it already exists"""
    if os.path.exists(vocab_file):
        print(f"-> Found the vocab file. Loading the vocabulary from {vocab_file}...")
        with open(vocab_file, 'r', encoding='utf-8') as f:
            vocab = json.load(f)
        print(f"-> Loaded vocabulary size: {len(vocab)}")
        return vocab

    print(f"-> File {vocab_file} not found. Building a new vocabulary...")
    all_words = []
    for cap in captions:
        all_words.extend(cap.split())
        
    word_counts = Counter(all_words)
    valid_words = [word for word, count in word_counts.items() if count >= min_freq]
    
    vocab = {"<pad>": PAD_IDX, "<start>": START_IDX, "<end>": END_IDX, "<unk>": UNK_IDX}
    for idx, word in enumerate(valid_words):
        vocab[word] = idx + 4 
        
    # Save the vocab for reuse
    with open(vocab_file, 'w', encoding='utf-8') as f:
        json.dump(vocab, f, ensure_ascii=False)
        
    print(f"-> Vocabulary size (words appearing >= {min_freq} times): {len(vocab)}")
    print(f"-> Saved the vocabulary to: {vocab_file}")
    return vocab

# ==========================================
# MODEL ARCHITECTURE
# ==========================================
class CLIPEncoder(nn.Module):
    def __init__(self, embed_dim):
        super(CLIPEncoder, self).__init__()
        self.clip_vision = CLIPVisionModel.from_pretrained(
            "openai/clip-vit-base-patch16",
            use_safetensors=True,
        )
        
        for param in self.clip_vision.parameters():
            param.requires_grad = False
            
        self.fc = nn.Linear(768, embed_dim)
        self.layer_norm = nn.LayerNorm(embed_dim)

    def forward(self, images):
        outputs = self.clip_vision(pixel_values=images)
        features = outputs.last_hidden_state 
        embeddings = self.fc(features) 
        return self.layer_norm(embeddings)

class PositionalEncoding(nn.Module):
    def __init__(self, embed_dim, dropout=0.1, max_len=100):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, embed_dim)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, embed_dim, 2).float() * (-math.log(10000.0) / embed_dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

class ImageCaptioningTransformer(nn.Module):
    def __init__(self, vocab_size, embed_dim, ff_dim, num_heads, num_layers):
        super(ImageCaptioningTransformer, self).__init__()
        self.encoder = CLIPEncoder(embed_dim)
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.pos_encoder = PositionalEncoding(embed_dim)
        
        decoder_layer = nn.TransformerDecoderLayer(d_model=embed_dim, nhead=num_heads, dim_feedforward=ff_dim, batch_first=True)
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.fc_out = nn.Linear(embed_dim, vocab_size, bias=False)
        self.fc_out.weight = self.embedding.weight

    def generate_square_subsequent_mask(self, sz):
        mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
        mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        return mask

    def forward(self, images, captions):
        memory = self.encoder(images) 
        tgt_embed = self.embedding(captions) 
        tgt_embed = self.pos_encoder(tgt_embed)
        tgt_mask = self.generate_square_subsequent_mask(captions.size(1)).to(images.device)
        tgt_key_padding_mask = (captions == PAD_IDX)
        out = self.transformer_decoder(tgt=tgt_embed, memory=memory, 
                                       tgt_mask=tgt_mask, 
                                       tgt_key_padding_mask=tgt_key_padding_mask)
        return self.fc_out(out)

def _unwrap_model(model):
    return model.module if isinstance(model, nn.DataParallel) else model

def _prepare_save_path(filepath):
    """Make sure the checkpoint is saved to a writable folder, not /kaggle/input."""
    if os.path.isdir(filepath):
        raise RuntimeError(
            f"'{filepath}' is a directory, not a checkpoint file. "
            "Pass CHECKPOINT_FILE or BEST_MODEL_FILE, e.g. /kaggle/working/last_checkpoint.pth."
        )
    save_dir = os.path.dirname(filepath) or "."
    os.makedirs(save_dir, exist_ok=True)
    if not os.access(save_dir, os.W_OK):
        raise RuntimeError(
            f"Cannot write the checkpoint to '{save_dir}'. "
            "On Kaggle, save to /kaggle/working and only load checkpoints from /kaggle/input."
        )
    return filepath

def _resolve_checkpoint_path(filepath):
    """Allow RESUME_CHECKPOINT_FILE to be a .pth file or a folder containing a checkpoint."""
    if os.path.isfile(filepath):
        return filepath

    if os.path.isdir(filepath):
        preferred_names = ["last_checkpoint.pth", "best_model.pth"]
        for name in preferred_names:
            candidate = os.path.join(filepath, name)
            if os.path.isfile(candidate):
                return candidate

        checkpoint_files = []
        for root, _, files in os.walk(filepath):
            for name in files:
                if name.endswith((".pth", ".pt")):
                    checkpoint_files.append(os.path.join(root, name))

        if checkpoint_files:
            checkpoint_files.sort(key=os.path.getmtime, reverse=True)
            return checkpoint_files[0]

        raise RuntimeError(f"'{filepath}' is a directory but contains no .pth/.pt file.")

    return filepath

class EarlyStoppingCallback:
    def __init__(self, patience=5, min_delta=0.0, save_path=BEST_MODEL_FILE):
        self.patience = patience
        self.min_delta = min_delta
        self.save_path = save_path
        self.counter = 0
        self.best_loss = float('inf')
        self.early_stop = False

    def __call__(self, val_loss, model, optimizer, epoch):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            self.save_best_model(model, optimizer, epoch)
        else:
            self.counter += 1
            print(f" EarlyStopping counter: {self.counter} / {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True

    def save_best_model(self, model, optimizer, epoch):
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': _unwrap_model(model).state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': self.best_loss
        }
        save_path = _prepare_save_path(self.save_path)
        torch.save(checkpoint, save_path)
        print(f" => Val Loss decreased. Saved BEST MODEL to: {save_path}")

# ==========================================
# CHECKPOINT MANAGEMENT AND TRAINING
# ==========================================
def save_checkpoint(model, optimizer, scheduler, epoch, val_loss, early_stopping, filepath=CHECKPOINT_FILE):
    """Save a checkpoint including the early-stopping state"""
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': _unwrap_model(model).state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler is not None else None,
        'val_loss': val_loss,
        'best_loss': early_stopping.best_loss,
        'es_counter': early_stopping.counter
    }
    filepath = _prepare_save_path(filepath)
    torch.save(checkpoint, filepath)
    print(f" => Saved the resume checkpoint to: {filepath}")

def load_checkpoint(model, optimizer=None, scheduler=None, early_stopping=None, filepath=CHECKPOINT_FILE, resume_training=True):
    """Load a checkpoint to continue training; if optimizer/scheduler are missing, load only the model weights."""
    start_epoch = 0
    if not resume_training:
        print("[*] RESUME_TRAINING = False. Training will start from epoch 1.")
        return start_epoch

    filepath = _resolve_checkpoint_path(filepath)

    if os.path.exists(filepath):
        print(f"[*] Found checkpoint '{filepath}'. Loading...")
        checkpoint = torch.load(filepath, map_location=device)
        state_dict = checkpoint.get('model_state_dict', checkpoint)
        _unwrap_model(model).load_state_dict(state_dict)

        if optimizer is not None and 'optimizer_state_dict' in checkpoint:
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

        if scheduler is not None and checkpoint.get('scheduler_state_dict') is not None:
            scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

        start_epoch = checkpoint.get('epoch', -1) + 1
        
        # Restore the early-stopping state
        if early_stopping is not None:
            early_stopping.best_loss = checkpoint.get('best_loss', float('inf'))
            early_stopping.counter = checkpoint.get('es_counter', 0)
        
        print(f"[*] Restored successfully. Training will continue from epoch {start_epoch + 1}")
        if early_stopping is not None:
            print(f"[*] EarlyStopping state: best_loss = {early_stopping.best_loss:.4f}, counter = {early_stopping.counter}")
    else:
        print(f"[*] Checkpoint '{filepath}' not found. Training will start from epoch 1.")
    return start_epoch

def train_model(train_loader, val_loader, vocab_size):
    model = ImageCaptioningTransformer(
        vocab_size=vocab_size, embed_dim=EMBED_DIM, 
        ff_dim=FF_DIM, num_heads=NUM_HEADS, num_layers=NUM_LAYERS
    ).to(device)
    
    if torch.cuda.is_available() and torch.cuda.device_count() > 1:
        print(f"[*] Using {torch.cuda.device_count()} GPUs with nn.DataParallel")
        model = nn.DataParallel(model)
        
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=LABEL_SMOOTHING)
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=0.01)
    
    num_training_steps = len(train_loader) * EPOCHS
    num_warmup_steps = int(0.05 * num_training_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_training_steps
    )
    
    amp_enabled = USE_AMP and torch.cuda.is_available()
    scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)
    print(f" Mixed precision (AMP): {'ON' if amp_enabled else 'OFF'} (USE_AMP={USE_AMP}, cuda={torch.cuda.is_available()})")

    early_stopping = EarlyStoppingCallback(patience=5, save_path=BEST_MODEL_FILE)
    
    # -------------------------
    # RESUME FROM CHECKPOINT
    # -------------------------
    start_epoch = load_checkpoint(
        model,
        optimizer,
        scheduler,
        early_stopping,
        filepath=RESUME_CHECKPOINT_FILE,
        resume_training=RESUME_TRAINING
    )
    
    log_interval = 200 # Large dataset, print less often to avoid flooding the screen
    
    for epoch in range(start_epoch, EPOCHS):
        # Seeding by epoch makes a resumed run reproduce the same batch/augmentation order.
        epoch_seed = SEED + epoch
        seed_everything(epoch_seed)
        if train_loader.generator is not None:
            train_loader.generator.manual_seed(epoch_seed)
        print(f"\n--- Starting epoch {epoch+1:02d}/{EPOCHS} ---")
        
        # -------------------------
        # PHASE 1: TRAINING
        # -------------------------
        model.train()
        train_loss = 0
        train_correct = 0
        train_tokens = 0
        
        for batch_idx, (images, captions) in enumerate(train_loader):
            images, captions = images.to(device), captions.to(device)
            
            decoder_input = captions[:, :-1]
            targets = captions[:, 1:]
            
            optimizer.zero_grad()

            with torch.cuda.amp.autocast(enabled=amp_enabled):
                outputs = model(images, decoder_input)
                loss = criterion(outputs.reshape(-1, vocab_size), targets.reshape(-1))
            
            predictions = outputs.argmax(dim=-1) 
            mask = (targets != PAD_IDX) 
            correct = (predictions == targets) & mask
            
            train_correct += correct.sum().item()
            train_tokens += mask.sum().item()
            
            scaler.scale(loss).backward()
            # unscale before clipping: clipping must use the real gradients, not the AMP-scaled ones.
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            train_loss += loss.item()
            
            if (batch_idx + 1) % log_interval == 0 or (batch_idx + 1) == len(train_loader):
                batch_acc = (correct.sum().item() / mask.sum().item()) * 100 if mask.sum().item() > 0 else 0.0
                print(f"   [Epoch {epoch+1:02d}] Batch {batch_idx+1:05d}/{len(train_loader)} | Batch Loss: {loss.item():.4f} | Batch Acc: {batch_acc:.2f}%")
                
        epoch_train_loss = train_loss / len(train_loader)
        epoch_train_acc = (train_correct / train_tokens) * 100
        
        # -------------------------
        # PHASE 2: VALIDATION
        # -------------------------
        model.eval()
        val_loss = 0
        val_correct = 0
        val_tokens = 0

        if val_loader.generator is not None:
            val_loader.generator.manual_seed(SEED + 100_000 + epoch)
        
        with torch.no_grad():
            for images, captions in val_loader:
                images, captions = images.to(device), captions.to(device)
                
                decoder_input = captions[:, :-1]
                targets = captions[:, 1:]
                
                outputs = model(images, decoder_input)
                loss = criterion(outputs.reshape(-1, vocab_size), targets.reshape(-1))
                val_loss += loss.item()
                
                predictions = outputs.argmax(dim=-1) 
                mask = (targets != PAD_IDX) 
                val_correct += ((predictions == targets) & mask).sum().item()
                val_tokens += mask.sum().item()

        epoch_val_loss = val_loss / len(val_loader)
        epoch_val_acc = (val_correct / val_tokens) * 100

        # -------------------------
        # REPORT AND SAVE CHECKPOINT
        # -------------------------
        print(f"\n=> EPOCH {epoch+1:02d} SUMMARY:")
        print(f"   Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.2f}%")
        print(f"   Val Loss  : {epoch_val_loss:.4f} | Val Acc  : {epoch_val_acc:.2f}%")
        
        # 1. Save the best model if the loss decreases
        early_stopping(epoch_val_loss, model, optimizer, epoch)
        
        # 2. Always save the last checkpoint after each epoch so training can resume if the Kaggle kernel dies
        save_checkpoint(model, optimizer, scheduler, epoch, epoch_val_loss, early_stopping, CHECKPOINT_FILE)
        
        if early_stopping.early_stop:
            print(f"⚠️ Early stopping at epoch {epoch+1}: Val Loss did not improve for several epochs.")
            break
            
    return model, optimizer

# ==========================================
# EXECUTION (MAIN)
# ==========================================
if __name__ == "__main__":
    print("0. Preparing the dataset and vocabulary for the COCO 2014 baseline...")
    print(f"   TRAIN_IMG_DIR  : {TRAIN_IMG_DIR}")
    print(f"   VAL_IMG_DIR    : {VAL_IMG_DIR}")
    print(f"   KARPATHY_JSON  : {KARPATHY_JSON_FILE}")
    
    # 0.1 Load the Train images and captions of the Karpathy split
    train_imgs, train_caps = load_karpathy_data(KARPATHY_JSON_FILE, split=TRAIN_SPLITS)
    
    # 0.2 Load the Val images and captions of the Karpathy split
    val_imgs, val_caps = load_karpathy_data(KARPATHY_JSON_FILE, split="val")
    
    # 0.3 Build (or load) the vocabulary from the Train split and save it to file (so it can be reloaded)
    vocab = build_or_load_vocab(train_caps, min_freq=MIN_WORD_FREQ, vocab_file=VOCAB_FILE)
    vocab_size = len(vocab)
    
    # 1. Create the Dataset & DataLoader
    train_dataset = COCODataset(train_imgs, train_caps, vocab, transforms=train_transforms)
    val_dataset = COCODataset(val_imgs, val_caps, vocab, transforms=val_transforms)

    print("1. Creating DataLoaders...")
    train_generator = torch.Generator()
    train_generator.manual_seed(SEED)
    val_generator = torch.Generator()
    val_generator.manual_seed(SEED + 1)
    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=4, persistent_workers=True, pin_memory=True if torch.cuda.is_available() else False,
        worker_init_fn=seed_worker, generator=train_generator
    )
    val_loader = DataLoader(
        val_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=4, persistent_workers=True, pin_memory=True if torch.cuda.is_available() else False,
        worker_init_fn=seed_worker, generator=val_generator
    )

    print(f"2. Training the CLIP-ViT-B/16 + Transformer baseline on COCO 2014...")
    model, optimizer = train_model(train_loader, val_loader, vocab_size=vocab_size)

    print("3. Training finished all requested epochs.")
